In [0]:
from pyspark.sql.functions import current_timestamp, col

# Leitura do arquivo CSV usando o Spark
df = (
    spark.read
        .format("csv")                     # Define o formato do arquivo como CSV
        .option("header", "true")           # Indica que a primeira linha do CSV contém o cabeçalho
        .option("sep", ",")                 # Define o separador de colunas como vírgula
        .load("/Volumes/techdados/bronze/vol_landing/products.csv")  # Caminho do arquivo CSV
        .withColumn(                       # Adiciona uma nova coluna ao DataFrame
            "_source_file",                # Nome da coluna
            col("_metadata.file_path")     # Captura o caminho do arquivo de origem (metadado do Spark)
        )
        .withColumn(                       # Adiciona outra coluna ao DataFrame
            "_ingestion_timestamp",        # Nome da coluna
            current_timestamp()            # Timestamp do momento da ingestão dos dados
        )
)

# Exibe o DataFrame na interface do Databricks (ou ambiente compatível)
display(df)


In [0]:
# Grava o DataFrame no formato Delta Lake
(
    df.write
        .option("overwriteSchema", "true")# Permite sobrescrever o schema
        .mode("overwrite")                # Sobrescreve os dados existentes
        .saveAsTable(
            "techdados.bronze.products"   # Nome da tabela (catálogo.schema.tabela)
        )
)

In [0]:
%sql
CREATE OR REPLACE TABLE techdados.bronze.products
USING DELTA
AS
SELECT
    *,                                   -- Todas as colunas do CSV
    _metadata.file_path       AS _source_file,          -- Caminho do arquivo de origem
    current_timestamp()       AS _ingestion_timestamp   -- Timestamp da ingestão
FROM read_files(
    '/Volumes/techdados/bronze/vol_landing/products.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);